In [ ]:
## in console to keep session alive
"""
function ClickConnect() {
  console.log("Checking connection status...");
  const colabButton = document.querySelector("colab-connect-button");
  if (colabButton && colabButton.shadowRoot) {
    const connectBtn = colabButton.shadowRoot.querySelector("#connect");
    if (connectBtn) {
      connectBtn.click();
      console.log("Connect button clicked ✅");
    } else {
      console.log("Connect button not found ❌");
    }
  } else {
    console.log("colab-connect-button element not found ❌");
  }
}
setInterval(ClickConnect, 60000);
"""

In [ ]:
#!pip install -U datasets
#!pip install -U huggingface_hub
#!pip install --upgrade datasets huggingface_hub
import transformers
print(transformers.__version__)

"""
SNLI + BERT + LSTM för Natural Language Inference (NLI)

Modellen fungerar enligt följande:

Premiss --------------------\
                             --> BERT --> LSTM --> Sammanfogning --> Klassificering
hypothesis --------------------/

Output:
0 = entailment
1 = contradiction
2 = neutral

Exempel:
Premiss: "A man is playing guitar."
hypothesis: "A person is making music."

=> entailment
"""

import datasets
import huggingface_hub

print(datasets.__version__)
print(huggingface_hub.__version__)

import torch
import torch.nn as nn

from datasets import load_dataset
from transformers import AutoTokenizer
from transformers import AutoModel

from torch.utils.data import Dataset
from torch.utils.data import DataLoader


In [ ]:



# CONFIGURATION


MODEL_NAME = "bert-base-uncased"

##MAX_LENGTH = 64
MAX_LENGTH = 128

##BATCH_SIZE = 16
BATCH_SIZE = 64

EPOCHS = 3

LEARNING_RATE = 2e-5

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# load SNLI


print("Laddar SNLI...")

#snli = load_dataset("snli") # this was wrong
snli = load_dataset("stanfordnlp/snli")
print(snli["train"])
"""
SNLI contains:

premise
hypothesis
label

Label:
0 : entailment
1 : neutral
2 = contradiction:
filtering rows with label -1
"""


## Filter invalid ones


def valid_example(example):
    return example["label"] != -1

train_data = snli["train"].filter(valid_example)

valid_data = snli["validation"].filter(valid_example)




# TOKENIZER


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)




In [ ]:

# bert-class


class SNLIDataset(Dataset):
    """
    Turn SNLI into  tensors.
    """

    def __init__(self, dataset):
        self.dataset = dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):

        row = self.dataset[idx]

        premise = row["premise"]
        hypothesis = row["hypothesis"]

        label = row["label"]

        # Tokenize premis
        prem_enc = tokenizer(
            premise,
            truncation=True,
            padding="max_length",
            max_length=MAX_LENGTH,
            return_tensors="pt"
        )

        ## Tokenize hypothesis
        hyp_enc = tokenizer(
            hypothesis,
            truncation=True,
            padding="max_length",
            max_length=MAX_LENGTH,
            return_tensors="pt"
        )

        return {
            "prem_input_ids":
                prem_enc["input_ids"].squeeze(0),

            "prem_attention":
                prem_enc["attention_mask"].squeeze(0),

            "hyp_input_ids":
                hyp_enc["input_ids"].squeeze(0),

            "hyp_attention":
                hyp_enc["attention_mask"].squeeze(0),

            "label":
                torch.tensor(label)
        }




In [ ]:

# DATALOADER


train_loader = DataLoader(
    SNLIDataset(train_data),
    batch_size=BATCH_SIZE,
    shuffle=True
)

valid_loader = DataLoader(
    SNLIDataset(valid_data),
    batch_size=BATCH_SIZE
)



In [ ]:

# model


class BertLSTMNLI(nn.Module):


    def __init__(self):

        super().__init__()

        # Förtränad BERT
        self.bert = AutoModel.from_pretrained(MODEL_NAME)

        bert_dim = self.bert.config.hidden_size
        # bert-base = 768

        # LSTM som arbetar på BERT-utdata
        self.lstm = nn.LSTM(
            input_size=bert_dim,
            hidden_size=256,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )



        # 256 * 2 = 512 as  LSTM is bidirectional
        sentence_dim = 512

        # Premiss + hypothesis
        combined_dim = sentence_dim * 2

        self.classifier = nn.Sequential(

            nn.Linear(combined_dim, 256),

            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(256, 3)
        )

    def encode_sentence(
        self,
        input_ids,
        attention_mask
    ):



        # BERT


        bert_output = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        """
        last_hidden_state:

        Shape:
        [batch, seq_len, 768]
        """

        sequence_output = bert_output.last_hidden_state


        # LSTM


        _, (hidden, _) = self.lstm(
            sequence_output
        )

        """
        hidden shape:

        [2, batch, 256]

        eftersom bidirectional=True

        hidden[0] = framåt
        hidden[1] = bakåt
        """

        forward_hidden = hidden[0]

        backward_hidden = hidden[1]

        # combining directions
        sentence_vector = torch.cat(
            [forward_hidden, backward_hidden],
            dim=1
        )

        return sentence_vector

    def forward(
        self,
        prem_input_ids,
        prem_attention,
        hyp_input_ids,
        hyp_attention
    ):

        #premiss
        prem_vector = self.encode_sentence(
            prem_input_ids,
            prem_attention
        )

        # hypothesis
        hyp_vector = self.encode_sentence(
            hyp_input_ids,
            hyp_attention
        )

        # Sammanfoga båda meningarna
        combined = torch.cat(
            [prem_vector, hyp_vector],
            dim=1
        )

        logits = self.classifier(combined)

        return logits



In [ ]:

# SKAPA MODELL



## testing stuff

print("BERT loaded")


model = BertLSTMNLI().to(DEVICE) # got problems
#!pip install transformers
#!pip install BertLSTMNLI
#from transformers import BertModel

#model = BertModel.from_pretrained("bert-base-uncased")
#print(model)


criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE
)


In [ ]:
### Training ###
## Accuracy with epochs
# 3 : 0,65
# 4 : 0,62
# 10:

##EPOCHS = 20

for epoch in range(EPOCHS):
    print(f"Beginning Epoch {epoch+1}/{EPOCHS}")
    model.train()

    total_loss = 0
    print("Batches: ", len(train_loader))
    counter = 1

    for batch in train_loader:

        if counter%10 == 0:
          pass
        print(f"Batch {counter}/{len(train_loader)}")
        counter += 1

        prem_ids = batch["prem_input_ids"].to(DEVICE)
        prem_att = batch["prem_attention"].to(DEVICE)

        hyp_ids = batch["hyp_input_ids"].to(DEVICE)
        hyp_att = batch["hyp_attention"].to(DEVICE)

        labels = batch["label"].to(DEVICE)

        optimizer.zero_grad()

        logits = model(
            prem_ids,
            prem_att,
            hyp_ids,
            hyp_att
        )

        loss = criterion(
            logits,
            labels
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(
        f"Ending Epoch {epoch+1}/{EPOCHS} "
        f"Loss: {avg_loss:.4f}"
    )


Beginning Epoch 1/3
Batches:  8584
Batch 1/8584


In [ ]:

# EVALUERING


model.eval()

correct = 0
total = 0

with torch.no_grad():

    for batch in valid_loader:

        prem_ids = batch["prem_input_ids"].to(DEVICE)
        prem_att = batch["prem_attention"].to(DEVICE)

        hyp_ids = batch["hyp_input_ids"].to(DEVICE)
        hyp_att = batch["hyp_attention"].to(DEVICE)

        labels = batch["label"].to(DEVICE)

        logits = model(
            prem_ids,
            prem_att,
            hyp_ids,
            hyp_att
        )

        predictions = torch.argmax(
            logits,
            dim=1
        )

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

accuracy = correct / total

print(f"\nVal Accuracy: {accuracy:.4f}")